In [1]:
from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd

from nibabel.processing import resample_to_output

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from resources.cut_off_edges import cut_off_edges
from resources.find_pitch_error_y_y import create_rotation_y_profile
from resources.find_rotation_error import create_rotation_x_profile
from resources.find_x import calculate_x_profile
from resources.standarize_image import standardize_image

In [2]:
project_dir = Path.cwd().parents[1]

# Jeżeli notebook znajduje się np. w katalogu src:
if not (project_dir / "data").exists():
    project_dir = project_dir.parent

niftis_dir = project_dir / "data" / "niftis"
labels_path = niftis_dir / "train_labels_JNDlMjr.csv"

outputs_dir = project_dir / "src" / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)

labels_df = pd.read_csv(labels_path)

target_column = "is_pathologic"

labels = labels_df.set_index("uid")[target_column]

nifti_paths = sorted(niftis_dir.glob("*.nii.gz"))

print("Katalog projektu:", project_dir)
print("Liczba obrazów:", len(nifti_paths))
print("Liczba etykiet:", len(labels))


Katalog projektu: C:\Users\abram\PycharmProjects\DrivenData_DAT_Parkinsons
Liczba obrazów: 1362
Liczba etykiet: 1362


In [3]:
uids = []
y_labels = []

x_profiles = []
rotation_x_profiles = []
rotation_y_profiles = []

for index, nifti_path in enumerate(nifti_paths, start=1):
    uid = nifti_path.name.removesuffix(".nii.gz")

    if uid not in labels.index:
        print(f"Brak etykiety dla {uid}")
        continue

    print(f"{index}/{len(nifti_paths)}: {uid}")

    image = nib.load(nifti_path)

    processed_image = cut_off_edges(image)

    processed_image = cut_off_edges(
        processed_image,
        x_left=35,
        x_right=35,
        y_front=40,
        y_back=40,
        z_down=30,
        z_up=30,
    )

    processed_image = standardize_image(
        processed_image,
        percent_top=0.02,
    )

    processed_image = resample_to_output(
        processed_image,
        voxel_sizes=(2.46, 2.46, 2.46),
        order=1,
    )

    processed_image = cut_off_edges(
        processed_image,
        x_left=30,
        x_right=30,
        y_front=30,
        y_back=30,
        z_down=20,
        z_up=25,
    )

    volume = processed_image.get_fdata(dtype=np.float32)

    profile_only_x = np.asarray(
        calculate_x_profile(volume),
        dtype=np.float32,
    )

    profile_rotation_x = np.asarray(
        create_rotation_x_profile(processed_image),
        dtype=np.float32,
    )

    profile_rotation_y = np.asarray(
        create_rotation_y_profile(processed_image),
        dtype=np.float32,
    )

    uids.append(uid)
    y_labels.append(int(labels.loc[uid]))

    x_profiles.append(profile_only_x)
    rotation_x_profiles.append(profile_rotation_x)
    rotation_y_profiles.append(profile_rotation_y)


1/1362: 01nouhtc
2/1362: 0224wk0y
3/1362: 049enulq
4/1362: 04x0qzfh
5/1362: 05xt54fn
6/1362: 06akt7zq
7/1362: 06nkwfg3
8/1362: 07gbasf5
9/1362: 07s387j5
10/1362: 085rntdr
11/1362: 08liuqdn
12/1362: 08uhbxlb
13/1362: 09lzrk79
14/1362: 09vs9kkh
15/1362: 0aeoorbt
16/1362: 0bq1s9ry
17/1362: 0dk0get8
18/1362: 0dl5e11e
19/1362: 0ezzf8s2
20/1362: 0fgzvrct
21/1362: 0i4x6g1e
22/1362: 0i6g61pc
23/1362: 0imjyjcm
24/1362: 0iv1ggxr
25/1362: 0j1oulwo
26/1362: 0jdcli69
27/1362: 0jf6dnob
28/1362: 0jjsp49c
29/1362: 0kb68rer
30/1362: 0l1offkh
31/1362: 0lzjjege
32/1362: 0odwmqx5
33/1362: 0pdw5py6
34/1362: 0qpop1o8
35/1362: 0s0r2sds
36/1362: 0sqcnf9w
37/1362: 0uc87nxp
38/1362: 0vgnxb5j
39/1362: 0vtkrrvw
40/1362: 0w10lbt3
41/1362: 0yknfu84
42/1362: 118qqvae
43/1362: 11bd2sce
44/1362: 11zsb8lv
45/1362: 12pgyme6
46/1362: 146vey0a
47/1362: 14nijcp5
48/1362: 153sflse
49/1362: 15ahdbvz
50/1362: 15ikpqah
51/1362: 15z8vmuh
52/1362: 167d6vnx
53/1362: 199dxj37
54/1362: 1b0z3nx3
55/1362: 1bj9rm7p
56/1362: 1c2zl46q
5

In [4]:
x_profiles = np.stack(x_profiles)
rotation_x_profiles = np.stack(rotation_x_profiles)
rotation_y_profiles = np.stack(rotation_y_profiles)

y_labels = np.asarray(y_labels, dtype=np.int8)
uids = np.asarray(uids)

In [5]:
output_path = project_dir / "data" / "processed_profiles.npz"

np.savez_compressed(
    output_path,
    uid=uids,
    is_pathologic=y_labels,
    x_profile=x_profiles,
    rotation_x_profile=rotation_x_profiles,
    rotation_y_profile=rotation_y_profiles,
)

In [6]:
data = np.load(
    project_dir / "data" / "processed_profiles.npz",
    allow_pickle=False,
)

uids = data["uid"]
y = data["is_pathologic"]

x_profile = data["x_profile"]
rotation_x_profile = data["rotation_x_profile"]
rotation_y_profile = data["rotation_y_profile"]
X = np.concatenate(
    [
        x_profile,
        rotation_x_profile,
        rotation_y_profile,
    ],
    axis=1,
)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", np.unique(y_train, return_counts=True))
print("y_test:", np.unique(y_test, return_counts=True))

X_train: (1089, 780)
X_test: (273, 780)
y_train: (array([0, 1], dtype=int8), array([492, 597]))
y_test: (array([0, 1], dtype=int8), array([123, 150]))


In [16]:

from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV

# pipeline = make_pipeline(
#     StandardScaler(),
#     LogisticRegression(
#         max_iter=20000,
#         random_state=42,
#     ),
# )
# cv = StratifiedKFold(
#     n_splits=5,
#     shuffle=True,
#     random_state=42,
# )
# param_grid = {
#     "logisticregression__C": [
#         0.001,
#         0.01,
#         0.1,
#         1,
#         10,
#         100,
#     ],
# }

pipeline = Pipeline([
    (
        "model",
        RandomForestClassifier(
            random_state=42,
            n_jobs=1,
        ),
    ),
])

param_distributions = {
    "model__n_estimators": [300, 500, 800, 1000],
    "model__max_depth": [None, 5, 10, 15, 20],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 4, 8],
    "model__max_features": ["sqrt", "log2", 0.5],
    "model__criterion": ["gini", "entropy", "log_loss"],
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=50,
    scoring="neg_log_loss",
    cv=cv,
    n_jobs=-1,
    random_state=42,
    verbose=2,
    return_train_score=True,
)

random_search.fit(X_train, y_train)

print("Najlepsze parametry:")
print(random_search.best_params_)

print("Najlepszy CV log loss:")
print(-random_search.best_score_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Najlepsze parametry:
{'model__n_estimators': 300, 'model__min_samples_split': 2, 'model__min_samples_leaf': 4, 'model__max_features': 'sqrt', 'model__max_depth': None, 'model__criterion': 'log_loss'}
Najlepszy CV log loss:
0.42691661613956117


In [20]:
best_model = random_search.best_estimator_

y_pred = best_model.predict(X_test)
y_probability = best_model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_probability))
print("Log loss:", log_loss(y_test, y_probability))

print("\nMacierz omyłek:")
print(confusion_matrix(y_test, y_pred))

print("\nRaport:")
print(classification_report(y_test, y_pred))

Accuracy: 0.8095238095238095
F1: 0.832258064516129
ROC AUC: 0.9071544715447154
Log loss: 0.3956266278373923

Macierz omyłek:
[[ 92  31]
 [ 21 129]]

Raport:
              precision    recall  f1-score   support

           0       0.81      0.75      0.78       123
           1       0.81      0.86      0.83       150

    accuracy                           0.81       273
   macro avg       0.81      0.80      0.81       273
weighted avg       0.81      0.81      0.81       273

